# SurviveCity GRPO Training — Colab Runner

Trains Qwen2.5-3B-Instruct with LoRA + GRPO on the SurviveCity zombie-survival environment.

## Setup before running

1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Click the 🔑 **Secrets** icon in the left sidebar and add two secrets:
   - `GITHUB_TOKEN` — GitHub PAT (fine-grained with `Contents: Read` on `SirjanSingh/zombiee`, or classic with `repo` scope)
   - `HF_TOKEN` — HuggingFace token with **write** scope
   - Make sure **Notebook access** is toggled ON for both
3. Edit `HUB_MODEL_ID` in the Configuration cell below to your HF repo
4. **Runtime → Run All**

⚠️ **If a cell fails, do `Runtime → Restart Session` and then `Run All` again.** The install step caches cleanly only after a restart if imports have already run.

## 1. Configuration

In [ ]:
HUB_MODEL_ID    = "noanya/zombiee"
REPO_URL        = "https://github.com/SirjanSingh/zombiee.git"
REPO_BRANCH     = "master"

MODEL_NAME       = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_STEPS        = 500
SAVE_STEPS       = 50
MAX_SEQ_LENGTH   = 2048
NUM_GENERATIONS  = 4
LR               = 1e-5
LORA_R           = 16
LORA_ALPHA       = 32
OUTPUT_DIR       = "/content/lora_v1"
REPO_DIR         = "/content/zombiee"
ENV_PORT         = 7860

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
print("Config loaded. HUB_MODEL_ID =", HUB_MODEL_ID)

## 2. GPU Sanity Check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free,compute_cap --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {cap[0]}.{cap[1]}")
    print(f"bf16 native: {cap[0] >= 8}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"Memory: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

## 3. Install Dependencies

Each `!pip` is its own shell invocation — no line-continuation, no whitespace traps. After this cell finishes the first time, do **Runtime → Restart Session** and then **Run All** again.

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q --upgrade "transformers>=4.47,<4.50" "accelerate>=1.2" "peft>=0.14" "datasets>=3.2" "trl>=0.12,<0.15" "bitsandbytes>=0.45"
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git" "unsloth_zoo"
!pip install -q hf_transfer "pydantic>=2.0" "fastapi>=0.104" "uvicorn[standard]>=0.24" "matplotlib>=3.8" tensorboard "huggingface_hub>=0.25"

import torch, transformers, bitsandbytes
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("CUDA OK:", torch.cuda.is_available())

## 4. Clone Private Repo & Install Package

Always wipes `REPO_DIR` first, then clones with the token passed via `http.extraheader` so the token never appears in `argv` or tracebacks. On error, the token is redacted before the message is raised.

In [ ]:
import os, subprocess, base64

try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
except Exception:
    tok = None
if not tok:
    import getpass
    tok = getpass.getpass("GitHub PAT: ")

subprocess.run(["rm", "-rf", REPO_DIR], check=False)

b64 = base64.b64encode(f"x-access-token:{tok}".encode()).decode()
auth = f"http.https://github.com/.extraheader=Authorization: Basic {b64}"

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
env["GIT_ASKPASS"] = "/bin/echo"

try:
    subprocess.run(
        ["git", "-c", auth, "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        env=env, check=True, capture_output=True, text=True,
    )
except subprocess.CalledProcessError as e:
    msg = ((e.stdout or "") + (e.stderr or "")).replace(tok, "***").replace(b64, "***")
    raise RuntimeError(f"git clone failed (rc={e.returncode}):\n{msg}") from None

subprocess.run(["pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

log = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout
print(log)
del tok, b64, auth
print("Repo cloned and installed")

## 5. HuggingFace Hub Login

In [ ]:
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass('HF token (write scope): ')

login(token=hf_token, add_to_git_credential=True)
os.environ['HUGGINGFACE_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token

api = HfApi()
try:
    api.create_repo(repo_id=HUB_MODEL_ID, exist_ok=True, private=False)
    print(f'Hub repo ready: https://huggingface.co/{HUB_MODEL_ID}')
except Exception as e:
    print(f'create_repo warning (ok if exists): {e}')

## 6. Detect Existing Checkpoints on Hub

In [ ]:
resume_arg = []
try:
    files = api.list_repo_files(HUB_MODEL_ID)
    has_ckpt = any(f.startswith('checkpoint-') or f == 'trainer_state.json' for f in files)
    if has_ckpt:
        print(f'Found existing artifacts in {HUB_MODEL_ID} -> will resume.')
        resume_arg = ['--resume-from-checkpoint', HUB_MODEL_ID]
    else:
        print(f'{HUB_MODEL_ID} is empty; starting fresh.')
except Exception as e:
    print(f'Could not list repo (probably empty): {e}; starting fresh.')
print('resume_arg =', resume_arg)

## 7. Env Smoke Test

In [ ]:
import sys, random
sys.path.insert(0, REPO_DIR)

from survivecity_env.env import SurviveCityEnv

env = SurviveCityEnv()
obs = env.reset(seed=42)
print(f"Reset OK: step={obs['step_count']}, reward={obs['reward']:.4f}, done={obs['done']}")
assert 0.01 <= obs['reward'] <= 0.99

actions = ['move_up', 'move_down', 'move_left', 'move_right', 'eat', 'wait']
steps = 0
while not obs.get('done', False) and steps < 350:
    aid = obs.get('metadata', {}).get('current_agent_id', 0)
    sc = obs.get('step_count', 0)
    if sc == 50:
        action = {'agent_id': aid, 'action_type': 'vote_lockout', 'vote_target': random.choice([0,1,2])}
    else:
        action = {'agent_id': aid, 'action_type': random.choice(actions)}
    obs = env.step(action)
    assert 0.01 <= obs['reward'] <= 0.99
    steps += 1

meta = obs.get('metadata', {})
print(f"Episode complete: {steps} actions, done={obs['done']}")
print(f"Final reward: {obs['reward']:.4f}, healthy_alive={meta.get('healthy_alive')}")
print(f"Postmortems: {len(meta.get('postmortems', []))}")
print('Environment smoke test PASSED')

## 8. Start Env Server

In [ ]:
import subprocess, time, urllib.request, sys, json

os.makedirs(OUTPUT_DIR, exist_ok=True)
env_log = open(f'{OUTPUT_DIR}/env_server.log', 'w')
env_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '127.0.0.1', '--port', str(ENV_PORT)],
    cwd=REPO_DIR, stdout=env_log, stderr=subprocess.STDOUT,
)

for i in range(30):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{ENV_PORT}/health', timeout=2)
        print(f'Env server up after {i+1}s (PID {env_proc.pid})')
        break
    except Exception:
        time.sleep(1)
else:
    env_log.flush()
    with open(f'{OUTPUT_DIR}/env_server.log') as f:
        print(f.read())
    raise RuntimeError('Env server did not start')

resp = urllib.request.urlopen(f'http://127.0.0.1:{ENV_PORT}/health')
health = json.loads(resp.read())
print(f'Health: {health}')
assert health == {'status': 'healthy'}

## 9. Run GRPO Training

Streams training output live. On T4 free tier, 500 steps takes ~3-4h. Checkpoints push to the HF Hub every 50 steps so you can resume on another machine if the session dies.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.train',
    '--env-url', f'http://127.0.0.1:{ENV_PORT}',
    '--model-name', MODEL_NAME,
    '--max-steps', str(MAX_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--num-generations', str(NUM_GENERATIONS),
    '--lr', str(LR),
    '--lora-r', str(LORA_R),
    '--lora-alpha', str(LORA_ALPHA),
    '--output-dir', OUTPUT_DIR,
    '--report-to', 'tensorboard',
    '--push-to-hub',
    '--hub-model-id', HUB_MODEL_ID,
    '--save-total-limit', '3',
] + resume_arg

print('Launching:', ' '.join(cmd))
print('=' * 60)
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    print('=' * 60)
    print(f'[Training exited with code {rc}]')
except KeyboardInterrupt:
    print('\nInterrupted; sending SIGTERM. Latest checkpoint is on the Hub.')
    proc.terminate()
    proc.wait()

## 10. Safety Net Upload

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f'Uploading {OUTPUT_DIR} -> {HUB_MODEL_ID}')
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        commit_message='final upload from colab',
        ignore_patterns=['_resume/*', 'env_server.log'],
    )
    print('Done')
else:
    print(f'Nothing to upload at {OUTPUT_DIR}')

## 11. Cleanup

In [ ]:
try:
    env_proc.terminate()
    env_proc.wait(timeout=10)
    print('Env server stopped')
except Exception as e:
    print('cleanup:', e)